In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

df = pd.read_csv('/content/drive/MyDrive/spamdata.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

df['label'] = df['label'].map({'ham': 0, 'spam': 1})

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]

    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]

    return ' '.join(words)

df['processed'] = df['message'].apply(preprocess_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['processed'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=3000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': LinearSVC()
}

for name, model in models.items():

    print(f"\n========== {name} ==========")

    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {acc:.4f}")

    print("\nClassification Report:")

    print(classification_report(
        y_test,
        y_pred,
        target_names=['Ham', 'Spam']
    ))

print("\n========== Spam Message Prediction ==========")

user_msg = input("Enter a message: ")

processed_msg = preprocess_text(user_msg)

msg_vector = vectorizer.transform([processed_msg])

prediction = models['Logistic Regression'].predict(msg_vector)

if prediction[0] == 1:
    print("\nThe message is SPAM")
else:
    print("\nThe message is HAM")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



========== Naive Bayes ==========
Accuracy: 0.9695

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.99      0.78      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


========== Logistic Regression ==========
Accuracy: 0.9677

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.98      0.77      0.86       149

    accuracy                           0.97      1115
   macro avg       0.97      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115


========== SVM ==========
Accuracy: 0.9848

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        S

### Predict on new input

In [6]:
df = pd.read_csv('/content/drive/MyDrive/spamdata.csv', encoding='latin-1')